# Compositional Contextual Expert Bandits [CCEB]

## Part 1: Defining The Inference System

In [ ]:
import torch
# Disable gradient calculations globally
torch.set_grad_enabled(False)
# Set so that each particle will only use one thread
torch.set_num_threads(1)
torch.set_num_interop_threads(1)
# set torch print options, including line width
torch.set_printoptions(precision=4, sci_mode=False, linewidth=250)

import numpy as np

import torch.distributions as D
import matplotlib.pyplot as plt
from copy import deepcopy

In [ ]:
from idealobserver import IdealObsPF
from models_context import CjCRP
from models_obs_rew import ConjugateGaussian, ConjugateOptimalArm, ConjugateBernoulli
from particle import ParticleParams
from experiment import ExperimentalEnv, IOPFSubjectHistory, ExperimentHistory, CombinedHistory

## Part 1: Defining the Experiment and Agent

### Define the various contexts that can be encountered

In [ ]:
NOISE = 0.02
cov_all = torch.eye(2)*NOISE

loc_c0_s0=torch.tensor([1.0, 0.0])
loc_c0_s1=torch.tensor([-1.0, 0.0])

loc_c1_s0=torch.tensor([0.0, 1.0])
loc_c1_s1=torch.tensor([0.0, -1.0])

context_o_params = {
    0: {  # context 0
        0: {'loc': loc_c0_s0, 'cov': cov_all},  # state 0
        1: {'loc': loc_c0_s1, 'cov': cov_all}   # state 1
    },
    1: {  # context 1
        0: {'loc': loc_c1_s0, 'cov': cov_all},  # state 0
        1: {'loc': loc_c1_s1, 'cov': cov_all}   # state 1
    }
}

context_r_params = {
    0: {  # context 0
        0: {'p_rew': torch.tensor([1.0, 0.0, 0.0, 0.0])},  # state 0
        1: {'p_rew': torch.tensor([0.0, 1.0, 0.0, 0.0])}   # state 1
    },
    1: {  # context 1
        0: {'p_rew': torch.tensor([0.0, 0.0, 1.0, 0.0])},  # state 0
        1: {'p_rew': torch.tensor([0.0, 0.0, 0.0, 1.0])}   # state 1
    }
}  

### Define an environment with these contexts

In [ ]:
env = ExperimentalEnv(context_o_params=context_o_params, context_r_params=context_r_params)

### Define the hyperparameters of the agent

In [ ]:
HYP_CJCRP = {
    'gamma' : 0.1,
    'alpha_o' : 0.5,
    'alpha_r' : 0.5,
}
HYP_NIW = {
    'mu0': torch.tensor([0.0, 0.0]),
    'kappa0': 0.1,
    'nu0': 4.0,
    'Lambda0': torch.eye(2) * 1.0 
}
HYP_BB = {
    'alpha0': torch.tensor([0.05, 0.05, 0.05, 0.05]),
    'beta0': torch.tensor([0.5, 0.5, 0.5, 0.5])
}

HYP_CAT = {
    'rho_c': 0.75,
    'rho_i': 0.25,
    'beta0': torch.tensor([0.25, 0.25, 0.25, 0.25])
}

### 2 Types of Agent

In [ ]:
Agent_Type1 : ParticleParams = {
    "type_context": CjCRP,
    "type_obs": ConjugateGaussian,
    "type_rew": ConjugateBernoulli,
    "hyp_context": HYP_CJCRP,
    "hyp_obs": HYP_NIW,
    "hyp_rew": HYP_BB
}

Agent_Type2 : ParticleParams = {
    "type_context": CjCRP,
    "type_obs": ConjugateGaussian,
    "type_rew": ConjugateOptimalArm,
    "hyp_context": HYP_CJCRP,
    "hyp_obs": HYP_NIW,
    "hyp_rew": HYP_CAT
}

## Part 2: Run Simulation

In [ ]:
AGENT_PARAMS = Agent_Type2

#### Simulation

In [ ]:
results = []

for i in range(32):

    # Initialize history buffers
    exp_history = ExperimentHistory()
    sub_history = IOPFSubjectHistory()


    # Initialize ensemble
    subject = IdealObsPF(N=250, hyp_params=AGENT_PARAMS, n_workers=8)
    #ensemble = Ensemble(N=1000, hyp_cjcrp=HYP_CJCRP, hyp_niw=HYP_NIW, hyp_bb=HYP_BB)

    # Run 50 trials in context (c_o=0, c_r=0)
    env.repeat_trials(subject, experiment_history=exp_history, subject_history=sub_history, N = 50, c_o=0, c_r=0, print_level=3)
    # Run 50 trials in context (c_o=1, c_r=1)
    env.repeat_trials(subject, experiment_history=exp_history, subject_history=sub_history, N = 50, c_o=1, c_r=1, print_level=3)
    # Run 50 trials in context (c_o=0, c_r=0)
    env.repeat_trials(subject, experiment_history=exp_history, subject_history=sub_history, N = 50, c_o=0, c_r=0, print_level=3)
    # Run 50 trials in context (c_o=0, c_r=1)
    env.repeat_trials(subject, experiment_history=exp_history, subject_history=sub_history, N = 50, c_o=0, c_r=1, print_level=3)

    subject.deallocate()

    # Combine histories into single object
    combined_history = CombinedHistory(subject_history=sub_history, experiment_history=exp_history)

    results.append(combined_history)

    
    plt.figure(figsize=(10, 3))
    plt.plot(combined_history.experiment_history.reward, label='Reward', color='blue', linewidth=2)
    plt.title('Reward Over Time')
    plt.xlabel('Time Step')
    plt.ylabel('Reward')
    plt.ylim(0, 1)
    plt.show()


## Part 3: Analysis of Results

### Analyse Classical and Compositional Savings

In [ ]:
proportion_optimal_mean = np.zeros(200)

chosen = list(range(32))

for j in chosen:

    proportion_optimal_mean += np.array(results[j].experiment_history.reward)

proportion_optimal_mean /= len(chosen)

# gaussian blur with kernel size 5 and sigma 1
from scipy.ndimage import gaussian_filter1d

def smooth(result):
    return gaussian_filter1d(result, sigma=1.0)

task_1 = smooth(proportion_optimal_mean[0:50])
task_1_dur = np.arange(0, 50)
task_2 = smooth(proportion_optimal_mean[50:100])
task_2_dur = np.arange(50, 100)
task_3 = smooth(proportion_optimal_mean[100:150])
task_3_dur = np.arange(100, 150)
task_4 = smooth(proportion_optimal_mean[150:200])
task_4_dur = np.arange(150, 200)



plt.figure(figsize=(10, 3))
#plt.plot(proportion_optimal_mean, label='Proportion Optimal Action', color='blue', linewidth=2)
plt.plot(task_1_dur, task_1, color='blue', linewidth=2, alpha=1.0)
plt.plot(task_2_dur, task_2, color='red', linewidth=2, alpha=1.0)
plt.plot(task_3_dur, task_3, color='blue', linewidth=2, alpha=1.0)
plt.plot(task_4_dur, task_4, color='indigo', linewidth=2, alpha=1.0)
plt.plot(task_3_dur, task_1, color='blue', linewidth=2, alpha=0.4)
plt.plot(task_4_dur, task_1, color='blue', linewidth=2, alpha=0.4)
plt.plot(task_4_dur, task_2, color='red', linewidth=2, alpha=0.4)
plt.title('Proportion of Optimal Actions Over Time')
plt.xlabel('Time Step')
plt.ylabel('Proportion')
plt.ylim(0, 1.0)
plt.xlim(0, len(proportion_optimal_mean))
#plt.savefig("g_proportion_optimal_3.pdf")
plt.show()

### Visualisation of Inferred Variables for a subject

In [ ]:
jump_times = [50, 100, 150]
j = 4
context_o_hist_arr = np.array(results[j].subject_history.context_o)
context_r_hist_arr = np.array(results[j].subject_history.context_r)
p_actions_arr = np.array(results[j].subject_history.p_action)
p_jump_arr = np.array(results[j].subject_history.p_jump)
p_state_arr = np.array(results[j].subject_history.p_state)
weight_arr = np.array(results[j].subject_history.weights)
p_action_optimal = results[j].p_optimal_action

print("Context O History:")
max_context_o = np.max(context_o_hist_arr) + 1
context_o_densities = np.zeros((len(context_o_hist_arr), max_context_o))
for t, element in enumerate(context_o_hist_arr):
    for particle_id, val in enumerate(element):
        context_o_densities[t, val] += weight_arr[t][particle_id]
context_o_densities = context_o_densities / context_o_densities.sum(axis=1, keepdims=True)

# set size of plot
plt.figure(figsize=(10, 3))
plt.imshow(context_o_densities.T, aspect='auto', cmap='viridis', interpolation='nearest')
#plt.colorbar(label='Density')
for jt in jump_times:
    plt.axvline(x=jt, color='red', linestyle='--', linewidth=2, alpha=0.7)


plt.xlabel('Time Step')
plt.ylabel('Context Index')
plt.title('Context O Density Over Time')
#plt.savefig("g_context_o_density.pdf")
plt.show()

print("Context R History:")
max_context_r = np.max(context_r_hist_arr) + 1
context_r_densities = np.zeros((len(context_r_hist_arr), max_context_r))
for t, element in enumerate(context_r_hist_arr):
    for particle_id, val in enumerate(element):
        context_r_densities[t, val] += weight_arr[t][particle_id]
context_r_densities = context_r_densities / context_r_densities.sum(axis=1, keepdims=True)

plt.figure(figsize=(10, 3))
plt.imshow(context_r_densities.T, aspect='auto', cmap='viridis', interpolation='nearest')
#plt.colorbar(label='Density')
for jt in jump_times:
    plt.axvline(x=jt, color='red', linestyle='--', linewidth=2, alpha=0.7)
plt.xlabel('Time Step')
plt.ylabel('Context Index')
plt.title('Context R Density Over Time')
#plt.savefig("g_context_r_density.pdf")
plt.show()

print("Actions History:")
plt.figure(figsize=(10, 3))
plt.imshow(p_actions_arr.T, aspect='auto', cmap='viridis', interpolation='nearest')
#plt.colorbar(label='Density')
plt.yticks(ticks=[0, 1, 2, 3], labels=['A0', 'A1', 'A2', 'A3'])
for jt in jump_times:
    plt.axvline(x=jt, color='red', linestyle='--', linewidth=2, alpha=0.7)
plt.xlabel('Time Step')
plt.ylabel('Action Index')
plt.title('Action Density Over Time')
#plt.savefig("g_action_density.pdf")
plt.show()

print("Jump History:")

plt.figure(figsize=(10, 3))
plt.plot(p_jump_arr, label='Jump', color='blue', linewidth=2)
plt.title('Jump Density Over Time')
plt.xlabel('Time Step')
plt.ylabel('Density')
plt.ylim(0, 1)
plt.xlim(0, len(p_jump_arr))
for jt in jump_times:
    plt.axvline(x=jt, color='red', linestyle='--', linewidth=2, alpha=0.7)
#plt.savefig("g_jump_density.pdf")
plt.show()